# Chapter 30 Coding Exam — CNN Auditing, Medical Patch Training & Saliency Attribution (100 Points)

**Book Reference:** Chapter 30, Section 30.7 (Coding Exam)  
**Suggested Time:** 45–55 minutes (Open Book, Closed Solution)

| Task | Medical Computer Vision Pipeline Component | Points |
| :--- | :--- | :---: |
| **Task 30.1** | `audit_cnn_feature_map_shapes(model, input_shape)` (Forward-hook shape auditor with cleanup) | 30 |
| **Task 30.2** | `train_medical_patch_cnn(n_samples=200, epochs=15, seed=42)` (Offline CNN training & AUC >= 0.90) | 35 |
| **Task 30.3** | `compute_saliency_attribution_map(model, x_single)` (Input-gradient lesion localization map) | 35 |
| **TOTAL** | | **100** |

In [ ]:
# @title Step 0: Click Play (▶) to Activate Chapter 30 Exam Grader & Helpers
!pip install -q sharukh-tutor
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import sharukh_tutor as tutor

class MedicalPatchCNN(nn.Module):
    """Provided reference architecture for Exam Task 30.2."""

    def __init__(self, num_classes=1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )
        self.fc = nn.Linear(16, num_classes)

    def forward(self, x):
        return self.fc(self.features(x))

def generate_synthetic_lesion_dataset(n_samples=160, img_size=16, seed=42):
    """Provided deterministic synthetic lesion patch generator."""
    rng = np.random.default_rng(seed)
    n_neg = n_samples // 2
    n_pos = n_samples - n_neg
    labels = np.array([0] * n_neg + [1] * n_pos, dtype=np.float32)
    rng.shuffle(labels)
    yy, xx = np.meshgrid(
        np.arange(img_size), np.arange(img_size), indexing="ij"
    )
    images = np.zeros((n_samples, 1, img_size, img_size), dtype=np.float32)
    for i, lbl in enumerate(labels):
        base = rng.uniform(0.15, 0.30)
        patch = rng.normal(base, 0.12, size=(img_size, img_size))
        if lbl == 1.0:
            cy = rng.integers(3, img_size - 3)
            cx = rng.integers(3, img_size - 3)
            amp = rng.uniform(0.95, 1.25)
            dist_sq = (yy - cy) ** 2 + (xx - cx) ** 2
            patch = patch + amp * np.exp(-dist_sq / (2.0 * 1.5 ** 2))
        images[i, 0] = np.clip(patch, 0.0, 2.0)
    return torch.from_numpy(images), torch.from_numpy(labels)

tutor.activate(chapter="ch30", notebook="coding_exam")


---
## Exam Task 30.1 (30 Points): `audit_cnn_feature_map_shapes(model, input_shape)`

Build a forward-hook shape auditor for CNNs:
1. Attach forward hooks to every `nn.Conv2d` and `nn.Linear` module discovered via `model.named_modules()`.
2. Switch `model.eval()` and execute `model(torch.zeros(input_shape))` inside `torch.no_grad()`.
3. Remove **all** registered hooks cleanly (use `try ... finally` so zero hooks ever leak).
4. Return a `list` of `(module_name, tuple(output.shape))` pairs in forward execution order.

In [ ]:
def audit_cnn_feature_map_shapes(model, input_shape):
    """Audit Conv2d and Linear output shapes via clean forward hooks."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    shapes = []
    handles = []

    def _make_hook(layer_name):
        def _hook(_mod, _inp, out):
            shapes.append((layer_name, tuple(out.shape)))
        return _hook

    for name, mod in model.named_modules():
        if isinstance(mod, (nn.Conv2d, nn.Linear)):
            handles.append(mod.register_forward_hook(_make_hook(name)))
    try:
        model.eval()
        with torch.no_grad():
            model(torch.zeros(input_shape))
    finally:
        for handle in handles:
            handle.remove()
    return shapes
    ### END SOLUTION


---
## Exam Task 30.2 (35 Points): `train_medical_patch_cnn(n_samples=200, epochs=15, seed=42)`

Train and evaluate `MedicalPatchCNN` end-to-end on CPU:
1. Generate `(X, y)` via `generate_synthetic_lesion_dataset(n_samples=n_samples, img_size=16, seed=seed)`.
2. Perform a stratified 75/25 train/validation split (`test_size=0.25, stratify=y.numpy(), random_state=seed`).
3. Call `torch.manual_seed(seed)`, instantiate `MedicalPatchCNN(num_classes=1)`, and train for `epochs` epochs with `nn.BCEWithLogitsLoss()` and `torch.optim.Adam(..., lr=0.01)` (mini-batch size 32).
4. Evaluate on the validation split in `model.eval()` + `torch.no_grad()` and return `{"val_acc": round4, "val_auc": round4, "final_train_loss": round5}` achieving `val_auc >= 0.90`.

In [ ]:
def train_medical_patch_cnn(n_samples=200, epochs=15, seed=42):
    """Train MedicalPatchCNN on synthetic lesions & return val metrics."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    X, y = generate_synthetic_lesion_dataset(
        n_samples=n_samples, img_size=16, seed=seed
    )
    idx = np.arange(len(y))
    tr_i, va_i = train_test_split(
        idx, test_size=0.25, stratify=y.numpy(), random_state=seed
    )
    X_tr, y_tr = X[tr_i], y[tr_i]
    X_va, y_va = X[va_i], y[va_i]

    torch.manual_seed(seed)
    model = MedicalPatchCNN(num_classes=1)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.BCEWithLogitsLoss()
    final_loss = 0.0

    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(X_tr))
        ep_loss = 0.0
        for start in range(0, len(X_tr), 32):
            b = perm[start : start + 32]
            opt.zero_grad()
            loss = loss_fn(model(X_tr[b]).squeeze(-1), y_tr[b])
            loss.backward()
            opt.step()
            ep_loss += loss.item() * len(b)
        final_loss = ep_loss / len(X_tr)

    model.eval()
    with torch.no_grad():
        va_probs = torch.sigmoid(model(X_va).squeeze(-1)).numpy()
    va_pred = (va_probs >= 0.5).astype(np.float32)
    val_acc = float((va_pred == y_va.numpy()).mean())
    val_auc = float(roc_auc_score(y_va.numpy(), va_probs))
    return {
        "val_acc": round(val_acc, 4),
        "val_auc": round(val_auc, 4),
        "final_train_loss": round(float(final_loss), 5),
    }
    ### END SOLUTION


---
## Exam Task 30.3 (35 Points): `compute_saliency_attribution_map(model, x_single)`

Compute an input-gradient pixel saliency map without mutating `x_single`:
1. Put `model.eval()` and clear existing parameter gradients (`model.zero_grad()`).
2. Clone `x_single` `(1, 1, H, W)` with `x = x_single.clone().detach().requires_grad_(True)`.
3. Compute `logit = model(x).squeeze()`, call `logit.backward()`, and extract `saliency = x.grad.detach().abs().squeeze().cpu().numpy()`.
4. Return `{"saliency_map": saliency, "peak_coord": (int_row, int_col), "pred_prob": round4}` where `pred_prob = round(float(torch.sigmoid(logit.detach()).item()), 4)`.

In [ ]:
def compute_saliency_attribution_map(model, x_single):
    """Compute |d(logit)/dx| saliency map, peak (row, col), and prob."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    model.eval()
    model.zero_grad()
    x = x_single.clone().detach().requires_grad_(True)
    logit = model(x).squeeze()
    logit.backward()
    saliency = x.grad.detach().abs().squeeze().cpu().numpy()
    peak_rc = np.unravel_index(int(np.argmax(saliency)), saliency.shape)
    peak_coord = (int(peak_rc[0]), int(peak_rc[1]))
    pred_prob = round(float(torch.sigmoid(logit.detach()).item()), 4)
    return {
        "saliency_map": saliency,
        "peak_coord": peak_coord,
        "pred_prob": pred_prob,
    }
    ### END SOLUTION


In [ ]:
exam_submissions = {
    "ch30_exam_01": audit_cnn_feature_map_shapes,
    "ch30_exam_02": train_medical_patch_cnn,
    "ch30_exam_03": compute_saliency_attribution_map,
}

tutor.check_exam("ch30", exam_submissions)


---
## Chapter 30 Exam Debugging Post-Mortem (Solution Notebook Only)

- **Task 30.1 (Leaked Forward Hooks):** Forgetting `handle.remove()` leaves hooks permanently attached to the model, silently appending tensors on every future training batch and causing memory leaks. Wrapping the dummy forward pass in `try ... finally` guarantees clean teardown.
- **Task 30.2 (Mini-Batch Updates & `BCEWithLogitsLoss`):** Full-batch gradient descent only executes 15 total weight updates in 15 epochs, whereas mini-batches of 32 execute ~75 updates and push validation ROC-AUC to `1.0000`. Always feed raw logits to `BCEWithLogitsLoss` and apply `torch.sigmoid` only when reporting probabilities.
- **Task 30.3 (Backprop From Logit & Non-Mutation):** Mutating `x_single.requires_grad_(True)` in place pollutes the caller's validation tensor, while backpropagating through `torch.sigmoid(logit)` suffers vanishing gradients when `pred_prob` is near `0.99`. Cloning first and calling `logit.backward()` keeps attribution crisp and side-effect free.